# FLARE ancestry-switch QC (GQ / DP)

Given a per-chromosome **annotated BCF/VCF** from `propagate_annotations`
(`GQ`, `DP`, `AN1`, `AN2`), this notebook:

1. Finds **ancestry switches** (haplotype `AN1`/`AN2` changes between consecutive sites)
2. Flags short **flickers** (A→B→A within `--flicker-max-bp`, default 50 kb)
3. Compares **GQ / DP** at switch sites vs background non-switch sites

CLI: `scripts/flare_switch_qc.py` (streams via `bcftools query`).

**Prerequisite:** run `PropagateAnnotations.wdl` for the chromosome of interest.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts"):
    if (_d / "terra_notebook.py").is_file():
        sys.path.insert(0, str(_d.resolve()))
        break
else:
    _bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not _bucket:
        raise FileNotFoundError(
            "scripts/ not found locally and WORKSPACE_BUCKET is unset. "
            "Upload scripts/ to gs://WORKSPACE/scripts/."
        )
    _dest = (Path.cwd() / "scripts").resolve()
    _dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        ["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_dest) + "/"]
    )
    sys.path.insert(0, str(_dest))

from terra_notebook import init_notebook

SCRIPTS = init_notebook("workspace_paths.py", "flare_switch_qc.py")
from workspace_paths import data_root

ROOT = data_root()
OUT = ROOT / "flare_switch_qc"
OUT.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)
print("OUT:", OUT)
print("scripts:", SCRIPTS)

## Config

Point `ANNOTATED_VCF` at the WDL output (`*.annotated.bcf`).
Optionally restrict to `analysis_samples.txt` from `tractor_01_prepare_inputs`.

In [ ]:
import os
from pathlib import Path

# GCS or local path to propagate-annotations output
ANNOTATED_VCF = os.environ.get(
    "ANNOTATED_VCF",
    "",  # e.g. gs://BUCKET/.../aou_lr_phase2_v1.chr22.annotated.bcf
)
assert ANNOTATED_VCF, "Set ANNOTATED_VCF env var or edit this cell"

# Optional: one sample ID per line. Empty → all VCF samples.
SAMPLES = os.environ.get("ANALYSIS_SAMPLES", "")

PREFIX = os.environ.get("SWITCH_QC_PREFIX", "chr22")
GQ_THRESHOLD = int(os.environ.get("GQ_THRESHOLD", "20"))
DP_THRESHOLD = int(os.environ.get("DP_THRESHOLD", "5"))
FLICKER_MAX_BP = int(os.environ.get("FLICKER_MAX_BP", "50000"))
BG_KEEP_EVERY = int(os.environ.get("BG_KEEP_EVERY", "50"))

# Smoke test: set e.g. 100000 to scan only the first N sites
MAX_SITES = os.environ.get("MAX_SITES")
MAX_SITES = int(MAX_SITES) if MAX_SITES else None

RUN_DIR = OUT / PREFIX
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_VCF = RUN_DIR / Path(ANNOTATED_VCF).name

print("ANNOTATED_VCF:", ANNOTATED_VCF)
print("SAMPLES:", SAMPLES or "(all)")
print("RUN_DIR:", RUN_DIR)
print(f"thresholds: GQ<{GQ_THRESHOLD}, DP<{DP_THRESHOLD}, flicker≤{FLICKER_MAX_BP} bp")

## Localize VCF (if GCS) and run switch scan

In [ ]:
import shutil
import subprocess
from pathlib import Path

assert shutil.which("bcftools"), "bcftools required on PATH"

src = ANNOTATED_VCF
if src.startswith("gs://"):
    if not LOCAL_VCF.is_file():
        print(f"gsutil cp {src} {LOCAL_VCF}")
        subprocess.check_call(["gsutil", "-m", "cp", src, str(LOCAL_VCF)])
        tbi = src + ".tbi"
        try:
            subprocess.check_call(["gsutil", "-m", "cp", tbi, str(LOCAL_VCF) + ".tbi"])
        except subprocess.CalledProcessError:
            print("no .tbi alongside VCF; continuing")
    vcf_path = LOCAL_VCF
else:
    vcf_path = Path(src)
    assert vcf_path.is_file(), vcf_path

# Confirm FORMAT tags exist
hdr = subprocess.check_output(["bcftools", "view", "-h", str(vcf_path)], text=True)
for tag in ("AN1", "AN2", "GQ", "DP"):
    assert f"ID={tag}," in hdr, f"missing FORMAT/INFO {tag} in {vcf_path}"
print("FORMAT tags OK")

cmd = [
    sys.executable,
    str(SCRIPTS / "flare_switch_qc.py"),
    "--vcf",
    str(vcf_path),
    "--out-dir",
    str(RUN_DIR),
    "--gq-threshold",
    str(GQ_THRESHOLD),
    "--dp-threshold",
    str(DP_THRESHOLD),
    "--flicker-max-bp",
    str(FLICKER_MAX_BP),
    "--bg-keep-every",
    str(BG_KEEP_EVERY),
]
if SAMPLES:
    samp = Path(SAMPLES)
    if str(SAMPLES).startswith("gs://"):
        local_samp = RUN_DIR / "samples.txt"
        subprocess.check_call(["gsutil", "cp", SAMPLES, str(local_samp)])
        samp = local_samp
    cmd.extend(["--samples", str(samp)])
if MAX_SITES is not None:
    cmd.extend(["--max-sites", str(MAX_SITES)])

print(" ".join(cmd))
subprocess.check_call(cmd)

## Load results

In [ ]:
import json
import pandas as pd

summary = json.loads((RUN_DIR / "summary.json").read_text())
switches = pd.read_csv(RUN_DIR / "switches.tsv.gz", sep="\t", compression="gzip")

print("sites scanned:", summary["sites_scanned"])
print("switches:", summary["n_switches"], "flickers:", summary["n_flicker_switches"])
print("\nEnrichment (switch / background):")
for k in ("enrichment_frac_low_gq", "enrichment_frac_low_dp", "enrichment_frac_rnc_I"):
    print(f"  {k}: {summary.get(k)}")

display(pd.DataFrame([
    {"set": "switch", **summary["switch_site_calls"]},
    {"set": "background", **summary["background_site_calls"]},
]))
display(pd.DataFrame([
    summary["switches_all"],
    summary["switches_flicker"],
    summary["switches_sustained"],
]))
switches.head()

## Are switches enriched for low GQ / DP?

If bad genotypes drive spurious LAI flips, switch sites should show **lower median GQ/DP**
and **higher fractions below threshold** than background, especially among **flickers**.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def load_col(name: str) -> np.ndarray:
    p = RUN_DIR / name
    if not p.is_file():
        return np.array([])
    return pd.read_csv(p)["value"].to_numpy()

sw_gq = load_col("switch_gq.csv")
bg_gq = load_col("background_gq.csv")
sw_dp = load_col("switch_dp.csv")
bg_dp = load_col("background_dp.csv")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, sw, bg, thr, title in (
    (axes[0], sw_gq, bg_gq, GQ_THRESHOLD, "GQ"),
    (axes[1], sw_dp, bg_dp, DP_THRESHOLD, "DP"),
):
    if len(sw) == 0 or len(bg) == 0:
        ax.set_title(f"{title}: no data")
        continue
    hi = max(np.percentile(sw, 99), np.percentile(bg, 99), thr * 2)
    bins = np.linspace(0, hi, 40)
    ax.hist(bg, bins=bins, density=True, alpha=0.45, label="background")
    ax.hist(sw, bins=bins, density=True, alpha=0.55, label="switch")
    ax.axvline(thr, color="k", ls="--", lw=1, label=f"threshold={thr}")
    ax.set_xlabel(title)
    ax.set_ylabel("density")
    ax.legend(frameon=False)
    ax.set_title(title)

fig.suptitle(f"Quality at ancestry switches vs background ({PREFIX})")
fig.tight_layout()
fig_path = RUN_DIR / "gq_dp_switch_vs_bg.png"
fig.savefig(fig_path, dpi=150)
print("wrote", fig_path)
plt.show()

## Flicker vs sustained switches

Flickers (A→B→A) are the pattern that looks like a bad local genotype tract.
Sustained switches may be real ancestry breakpoints.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, col, thr, title in (
    (axes[0], "gq", GQ_THRESHOLD, "GQ at switch"),
    (axes[1], "dp", DP_THRESHOLD, "DP at switch"),
):
    for label, mask in (
        ("flicker", switches["is_flicker"] == 1),
        ("sustained", switches["is_flicker"] == 0),
    ):
        vals = switches.loc[mask, col].dropna().to_numpy()
        if len(vals) == 0:
            continue
        hi = max(np.percentile(vals, 99), thr * 2)
        bins = np.linspace(0, hi, 30)
        ax.hist(vals, bins=bins, density=True, alpha=0.5, label=f"{label} (n={len(vals)})")
    ax.axvline(thr, color="k", ls="--", lw=1)
    ax.set_xlabel(col.upper())
    ax.set_title(title)
    ax.legend(frameon=False)

fig.tight_layout()
fig_path = RUN_DIR / "gq_dp_flicker_vs_sustained.png"
fig.savefig(fig_path, dpi=150)
print("wrote", fig_path)
plt.show()

print("flicker median GQ:", switches.loc[switches.is_flicker == 1, "gq"].median())
print("sustained median GQ:", switches.loc[switches.is_flicker == 0, "gq"].median())

## Top samples / regions by switch density

In [ ]:
by_sample = (
    switches.groupby("sample")
    .agg(
        n_switches=("pos", "size"),
        n_flicker=("is_flicker", "sum"),
        median_gq=("gq", "median"),
        median_dp=("dp", "median"),
        frac_low_gq=("gq", lambda s: (s < GQ_THRESHOLD).mean()),
    )
    .sort_values("n_switches", ascending=False)
)
display(by_sample.head(20))

# 1 Mb bins
switches = switches.copy()
switches["mb"] = switches["pos"] // 1_000_000
by_mb = (
    switches.groupby(["chrom", "mb"])
    .agg(n_switches=("pos", "size"), n_flicker=("is_flicker", "sum"), median_gq=("gq", "median"))
    .sort_values("n_switches", ascending=False)
)
display(by_mb.head(20))

by_sample.to_csv(RUN_DIR / "switches_by_sample.tsv", sep="\t")
by_mb.to_csv(RUN_DIR / "switches_by_mb.tsv", sep="\t")

## Interpretation checklist

| Observation | Suggests |
|-------------|----------|
| Switch sites ≪ background median GQ/DP; enrichment ≫ 1 | Bad genotypes associated with flips |
| Flickers worse GQ/DP than sustained | Spurious local flips (your hypothesis) |
| High `frac_rnc_I` at switches | DeepVariant read no-calls at flip loci |
| Switches clustered in telomeres / low-mappability | Hard regions, not necessarily GQ alone |
| Enrichment ≈ 1 | Switches not explained by GQ/DP; look at phasing / FLARE params |

Outputs in `RUN_DIR`:
- `switches.tsv.gz` — every switch event
- `summary.json` — enrichment stats
- `gq_dp_*.png` — figures
- `switches_by_sample.tsv`, `switches_by_mb.tsv`